# ZeroID as ODIS Layers 1–2 — the SDK companion

Companion to [`odis-walkthrough.ipynb`](./odis-walkthrough.ipynb) (raw HTTP, wire-format
view) and the [role-capability statement](../../docs/odis/role-capability-statement.md).
This notebook replays the same ODIS story through the **Python SDK**
(`pip install highflame`), which shifts the vantage point in one important way:

- The raw notebook shows the **authorization server** enforcing ODIS at issuance.
- This notebook also shows the **ODIS-aware target** (ODIS §2.5 "native-mode downstream
  path", L2-15): `client.tokens.verify()` validates the Agent Runtime Credential *locally*
  against the JWKS — exactly what a native-mode resource server does — and the returned
  identity carries typed guards (`require_scope`, `require_trust`, `is_delegated`) that a
  tool server calls before executing anything.

**Prerequisites** (repo root): `make setup-keys && docker compose up -d`, then
`pip install "highflame==0.3.17" pyjwt cryptography` — 0.3.17 is the SDK version these
committed outputs were generated against.

> **Configuration note (ODIS-L1-09):** this notebook requires the compose default
> `token.require_dpop: false`. The Python SDK cannot construct DPoP proofs yet
> (highflame-sdk#105), so under the hardened configuration the role-capability statement
> grades as L1-09 *Meets (via configuration)*, every issuance call below is refused with
> `invalid_dpop_proof`. The DPoP-bound path is demonstrated over raw HTTP in the main
> walkthrough.
>
> **Local dev trust model:** admin-plane SDK calls (`identities.update`, `signals.ingest`,
> …) authenticate here only by client-supplied tenant IDs — a dev-mode convenience.
> Production fronts the admin plane with authenticated sessions and derives tenancy from
> verified claims.

In [1]:
import time, uuid, jwt
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import serialization
from highflame.zeroid import ZeroIDClient

client = ZeroIDClient(base_url="http://localhost:8899",
                      account_id="acct-sdk-demo", project_id="proj-sdk-demo")
run = uuid.uuid4().hex[:6]
print(client.health(), "| run id:", run)

status='healthy' service='zeroid' timestamp=datetime.datetime(2026, 9, 3, 23, 22, 40, 801620, tzinfo=TzInfo(0)) uptime_ms=56672 | run id: 1a5805


## 1 · Agent Registration Record — Layer 1 · The Passport (ODIS §6.1)

`agents.register` creates the durable governance record and returns the bootstrap API key
in one call. The agent is born `unverified` with a stable WIMSE URI (ODIS's `agent_id`).

> **SDK coverage note (0.3.17):** `client.credential_policies` can create issuance-ceiling
> policies, but there is no SDK parameter yet to *attach* one to an identity — the
> policy-gated fail-closed issuance demo therefore lives in the raw-HTTP companion. Worth
> an upstream SDK issue; the REST API supports it (`credential_policy_id`).

In [2]:
reg = client.agents.register(name="Orchestrator", external_id=f"orch-{run}",
                             sub_type="orchestrator", created_by="demo-admin@example.com")
orch = reg.identity
client.identities.update(orch.id, allowed_scopes=["data:read", "data:write"])
print(orch.wimse_uri, "| trust:", orch.trust_level, "| status:", orch.status)

spiffe://highflame.ai/acct-sdk-demo/proj-sdk-demo/agent/orch-1a5805 | trust: unverified | status: active


## 2 · Agent Runtime Credential — the Passport's output (ODIS §6.2, L1-05)

Short-lived, scoped issuance from the bootstrap key. (The DPoP holder-binding variant is in
the raw companion; the SDK issues bearer tokens as of 0.3.17.)

In [3]:
token = client.tokens.issue_api_key(reg.api_key, scope="data:read data:write")
print(token.token_type, "| expires_in:", token.expires_in, "s | scope:", token.scope)

Bearer | expires_in: 3600 s | scope: data:read data:write


## 3 · The ODIS-aware target — the Bridge's native mode (ODIS §2.5 / L2-15)

ODIS §2.5 defines a native-mode target as one that independently validates the Agent
Runtime Credential and the active Delegation Record and enforces their **audience, holder
binding, attenuation, constraints, freshness, and revocation** semantics. This section
demonstrates the validation + attenuation/trust slice of that duty list; §5 covers
revocation. Audience enforcement is demonstrated below; holder-binding verification at the
target (checking a DPoP proof against `cnf.jkt`) is what ZeroID's Go `pkg/dpop` provides
resource servers — not yet surfaced in the Python SDK.

`tokens.verify()` performs the local JWKS validation — no round-trip to the authorization
server. One caution before leaning on it: local verification is **revocation-blind** — a
token the server has already revoked keeps verifying until it expires. §5 demonstrates that
divergence and what a native-mode target must add. The typed guards are the enforcement. Note the
trust gate fires at the *target*, complementing the issuance-side gate the raw notebook
shows: the agent holds a perfectly valid token, and the tool server still refuses it until
the identity is trusted enough.

In [4]:
identity = client.tokens.verify(token.access_token)   # local JWKS validation
print("verified:", identity.sub, "| delegated:", identity.is_delegated())

identity.require_scope("data:read")                    # passes silently
for check in (lambda: identity.require_scope("admin:all"),
              lambda: identity.require_trust("first_party")):
    try:
        check()
    except Exception as e:
        print(f"refused ({type(e).__name__}):", e)

verified: spiffe://highflame.ai/acct-sdk-demo/proj-sdk-demo/agent/orch-1a5805 | delegated: False
refused (ZeroIDError): Missing required scope: 'admin:all'
refused (ZeroIDError): Insufficient trust level: required 'first_party', got 'unverified'


Audience is the first duty on ODIS §2.5's native-mode list, and it is one argument
here: a target passes the audience it serves, and `verify` refuses tokens minted for anyone
else. (ZeroID stamps `aud` on every credential per JWT-SVID §3, defaulting to the issuer URL
when the requester names no audience.)

In [5]:
aud = jwt.decode(token.access_token, options={"verify_signature": False})["aud"]  # display-only peek
print("token aud:", aud)
client.tokens.verify(token.access_token, audience=aud[0])       # expected audience: passes
try:
    client.tokens.verify(token.access_token, audience="https://some-other-service.example")
except Exception as e:
    print(f"wrong audience refused ({type(e).__name__}):", e)

token aud: ['http://localhost:8899']
wrong audience refused (ZeroIDError): Token verification failed: Audience doesn't match


Trust is **minted into the credential at issuance** — `trust_level` is a claim. So
promotion never upgrades outstanding tokens: the agent re-issues to benefit. The corollary
is the one that matters for security: *demotion doesn't downgrade them either* — a demoted
identity's already-issued credentials keep their elevated claim until expiry, which is why a
trust downgrade must travel with revocation (§5's compromise signal), not just a record
update. Here the identity gets promoted (via the admin surface for brevity — the *attested*
promotion path, ODIS-L1-03/11, is demonstrated in the raw companion), and a **fresh**
credential now clears the target's trust gate:

In [6]:
client.identities.update(orch.id, trust_level="first_party")
token = client.tokens.issue_api_key(reg.api_key, scope="data:read data:write")
identity = client.tokens.verify(token.access_token)
identity.require_trust("first_party")
print("trust gate cleared:", identity.trust_level)

trust gate cleared: first_party


## 4 · Delegation Record — Layer 2 · The Bridge (ODIS §6.3, Pillar 4, L2-05/06)

The researcher registers with its own holder key, proves possession of it in the
`actor_token` (a self-signed ES256 assertion — the SDK doesn't mint these; five lines of
PyJWT do), and the orchestrator delegates via RFC 8693. Attenuation is **monotonic and
visible**: the researcher's ceiling is `data:read`, so requesting `data:read data:write`
yields a token silently narrowed to `data:read` — and a request where *nothing* survives
the intersection is refused outright. (With a credential policy attached, any
out-of-ceiling request errors instead of narrowing — the stricter mode is in the raw
companion.)

In [7]:
priv = ec.generate_private_key(ec.SECP256R1())
pub_pem = priv.public_key().public_bytes(serialization.Encoding.PEM,
    serialization.PublicFormat.SubjectPublicKeyInfo).decode()
researcher = client.identities.create(
    external_id=f"researcher-{run}", owner_user_id="demo-admin@example.com",
    name="Researcher", identity_type="agent", sub_type="tool_agent",
    trust_level="first_party", allowed_scopes=["data:read"], public_key_pem=pub_pem)

now = int(time.time())
actor_token = jwt.encode({"iss": researcher.wimse_uri, "sub": researcher.wimse_uri,
                          "aud": ["http://localhost:8899"], "iat": now, "exp": now + 300},
                         priv, algorithm="ES256")

delegated = client.tokens.issue_token_exchange(
    subject_token=token.access_token, actor_token=actor_token, scope="data:read data:write")
print("granted scope (requested read+write):", repr(delegated.scope))

d_identity = client.tokens.verify(delegated.access_token)
print("sub:", d_identity.sub.split("/")[-1], "| is_delegated:", d_identity.is_delegated(),
      "| act:", d_identity.act, "| depth:", d_identity.delegation_depth)

try:
    client.tokens.issue_token_exchange(subject_token=token.access_token,
                                       actor_token=actor_token, scope="data:write admin:all")
except Exception as e:
    print(f"empty intersection refused ({type(e).__name__}):", e)

granted scope (requested read+write): 'data:read'
sub: researcher-1a5805 | is_delegated: True | act: {'sub': 'spiffe://highflame.ai/acct-sdk-demo/proj-sdk-demo/agent/orch-1a5805'} | depth: 1
empty intersection refused (APIError): [400] invalid_scope: requested scopes are not available for delegation


## 5 · Compromise signal, cascade — and what native mode must add — L1-12, L3-04/05

A `critical` CAE signal against the orchestrator kills its credentials and cascades to the
researcher's delegated token. Then the ODIS-relevant subtlety: **local JWKS verification
cannot see revocation** — the delegated token still verifies offline, while introspection
reports it dead. ODIS's definition of an ODIS-aware target requires enforcing *revocation
semantics*, not just signatures: a native-mode target pairs local verification with
introspection, short cache windows, or the revocation event stream (`GET /signals/stream`).
This is exactly the trade the role-capability statement flags under L2-11/L3-04.

In [8]:
client.signals.ingest(signal_type="anomalous_behavior", source="sdk-demo",
                      identity_id=orch.id, severity="critical",
                      payload={"reason": "prompt injection detected"})

print("introspection:", client.tokens.introspect(delegated.access_token).active)
still_verifies = bool(client.tokens.verify(delegated.access_token).sub)
print("local JWKS verify alone:", still_verifies, "← why native-mode targets must also check revocation state")

introspection: False
local JWKS verify alone: True ← why native-mode targets must also check revocation state


## What the SDK view added

| ODIS concept | Where |
|---|---|
| Native-mode target validation (§2.5, L2-15) | §3 — local JWKS verify + typed guards |
| Audience enforcement (§2.5 duty list) | §3 — `verify(audience=...)` refusing a mis-audienced token |
| Target-side trust gating (complement to issuance gating) | §3 — `require_trust` refusing a valid token |
| Monotonic attenuation, both modes (L2-06) | §4 — silent narrowing + empty-intersection refusal |
| Revocation semantics are part of native mode (L3-04, L2-11) | §5 — verify-vs-introspect divergence |

**SDK gaps observed (0.3.17), worth upstream issues (tracked in sdk#105):** no
`credential_policy_id` attachment on identities/agents, no attestation submit/verify, no
delegation-graph reads, and no DPoP proof support on token issuance — now load-bearing,
since the server can refuse proof-less issuance deployment-wide (`token.require_dpop`,
zeroid#304) — nor resource-server-side holder-binding (`cnf.jkt`/DPoP-proof) verification,
so a Python native-mode target can enforce signature, expiry, scope, trust, delegation,
audience, and (via introspection) revocation, but not yet holder binding. Everything above that the SDK lacks is
demonstrated over raw HTTP in the [main walkthrough](./odis-walkthrough.ipynb).